# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [2]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [3]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Wed Nov 11 07:28:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [4]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [5]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [6]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [7]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 And why is it so important?
Quantum computing is the process of using quantum phenomena to perform computation. It’s one of those things that sounds like science fiction, but is in fact real and available to you right now. The computing power of quantum computers is orders of magnitude larger than the most powerful computers today. As a result, they can solve problems that are intractable for conventional computers. They can also be used to simulate quantum systems, allowing scientists to study complex phenomena like quantum entanglement and superposition.
There are many potential applications for quantum computers, including:
- Drug discovery: Quantum computers can be used to simulate molecular interactions, allowing scientists to design new drugs more efficiently.
- Financial modeling: Quantum computers can be used to model

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a revolutionary field of computer science that leverages the principl

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [8]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To find the product of 15 and 24, we can use the standard multiplication algorithm. Here's how we can do it step by step:

1. Write down the numbers to be multiplied:

```
   15
×  24
```

2. Multiply 15 by the units digit of 24, which is 4:

```
   15
×  24
------
   60
```

3. Multiply 15 by the tens digit of 24, which is 2, and place the result one position to the left:

```
   15
×  24
------
   60
  300
------
```

4. Add the two partial products together:

```
   15
×  24
------
   60
  300
------
  360
```

So, the product of 15 and 24 is 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To determine how much flour is needed for 30 cookies, we can start by finding the amount of flour required for a single cookie. 

The recipe calls for 2 cups of flour for 12 cookies. To find th

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [10]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [11]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [12]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [13]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [14]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [15]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

---
# Відповіді на питання лабораторної 5 (п. 0.1, питання 11–21)

> Числа й виводи в цих відповідях взято з мого запуску в Colab (GPU Tesla T4). Теоретичні пояснення від конкретного запуску не залежать. Окремо зазначено там, де факт у виводі відрізнявся від очікуваного.

### 11. Як довго виконувалися клітинки? Чи були проблеми з запуском?

**Час.** Точних секундомірів клітинок у збереженому файлі немає (Colab їх не зберіг), тому загальну цифру не наводжу. Найтривалішими були завантаження двох моделей SmolLM3-3B
(base і instruct, ≈ 6 GB кожна) і генерація: у моїй клітинці для питання 16 режим `/think` генерував до 1000 токенів на задачу. Решта клітинок (шаблони, токенізація) — секунди.
GPU: Tesla T4.

**Проблеми.** Помилок не було, усі клітинки виконались, а обидві моделі по 3 млрд параметрів вмістились у памʼять T4 без OOM. У виводах є лише попередження, які на результат не впливають:
про неавторизовані запити до Hugging Face Hub (немає `HF_TOKEN`); про параметр `clean_up_tokenization_spaces`, який ігнорується для BPE-токенізатора; про поле `rope_parameters`
в конфігурації Qwen3.

### 12. Для чого змінювати `add_generation_prompt`? Як за згенерованими розмовами визначити його значення?

- `add_generation_prompt=False` — діалог **завершений**. Так форматують тренувальні дані: відповідь асистента вже є в списку повідомлень.
- `add_generation_prompt=True` — до кінця рядка додається **заголовок відповіді асистента** (`<|im_start|>assistant`). Це сигнал
  «тепер твоя черга говорити»; без нього модель могла б продовжити репліку користувача або почати довільний текст. Використовується для інференсу.

**Як визначити значення за результатом.** Дивимось на кінець рядка. З `True` він закінчується відкритим заголовком асистента
(без тексту відповіді й без закриваючого `<|im_end|>`). З `False` він закінчується закритим останнім повідомленням (`<|im_end|>`).
Клітинка нижче друкує кінці обох варіантів для всіх діалогів.

In [16]:
for conv_type, messages in conversations.items():
    for flag in (False, True):
        s = instruct_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=flag)
        print(f"{conv_type:<15} add_generation_prompt={str(flag):<5} …{s[-70:]!r}")
    print()

simple_qa       add_generation_prompt=False …'y Hugging Face.\n\n<|im_start|>user\nWhat is machine learning?<|im_end|>\n'
simple_qa       add_generation_prompt=True  …'s machine learning?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n'

with_system     add_generation_prompt=False …'ncepts clearly.\n\n<|im_start|>user\nWhat is machine learning?<|im_end|>\n'
with_system     add_generation_prompt=True  …'s machine learning?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n'

multi_turn      add_generation_prompt=False …'im_end|>\n<|im_start|>user\nCan you give me a simple example?<|im_end|>\n'
multi_turn      add_generation_prompt=True  …'e a simple example?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n'

reasoning_task  add_generation_prompt=False …'in travels 120 miles in 2 hours, what is its average speed?<|im_end|>\n'
reasoning_task  add_generation_prompt=True  …'n 2 hours, what is its average speed?<|im_end|>\n<|im_start|>assistant\n'



**Мій вивід** (кінці рядків для `simple_qa`):

- `add_generation_prompt=False`: `…<|im_start|>user\nWhat is machine learning?<|im_end|>\n`. Рядок закінчується **закритим** повідомленням користувача.
- `add_generation_prompt=True`: `…<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n`. Додано **відкритий заголовок асистента**, а в режимі `/no_think` ще й порожній блок `<think></think>`.

Так само виглядають `with_system` і `multi_turn`. У `reasoning_task` (режим `/think`) з `True` рядок закінчується просто `<|im_start|>assistant\n`, без блоку `<think>`:
порожній блок міркувань додається лише в режимі `/no_think`.

### 13. Чим відрізняються ролі `system` і `assistant`?

- **`system`** — інструкції розробника: хто ця модель, як відповідати, які є обмеження (у SmolLM3 тут ще й прапорець `/think` або `/no_think`).
  Це «налаштування» розмови: стоїть на початку, зазвичай один раз, діє на весь діалог, користувач його, як правило, не бачить.
  Модель не має його генерувати.
- **`assistant`** — відповіді самої моделі. У історії діалогу це її попередні репліки, а під час SFT — це **цільовий текст**,
  який модель має навчитися відтворювати (саме на цих токенах рахується loss при `assistant_only_loss=True`).

Коротко: `system` каже, *як* відповідати, `assistant` — це *сама відповідь*. Одне задається ззовні, інше породжується моделлю.

### 14. Вивід секції 2: які секції додав шаблонізатор і для чого?

У моєму виводі для `simple_qa` (системне повідомлення було лише `/no_think`, повідомлення користувача — «What is machine learning?») шаблон SmolLM3 побудував (скорочено):

```
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
```

Що додав шаблон і навіщо:

1. **Спеціальні токени `<|im_start|>роль … <|im_end|>`** (формат ChatML) навколо повідомлень: вони показують моделі межі реплік і хто говорить.
   Цікаво, що системний блок у SmolLM3 у моєму виводі **не закривається** `<|im_end|>`: він закінчується порожнім рядком перед `<|im_start|>user`.
2. **Блок `## Metadata`:** дата відсічення знань (June 2025), **поточна дата** (11 November 2026: підставилась автоматично, ми її не писали) і режим міркування.
3. **Прапорець режиму.** Ми написали в system лише `/no_think`; шаблон прибрав його з тексту й переніс у `Reasoning Mode: /no_think`. Якщо прапорець стояв поруч з іншим текстом
   («You are a math tutor. /no_think»), у `Custom Instructions` лишається тільки «You are a math tutor.».
4. **Стандартна інструкція** `You are a helpful AI assistant named SmolLM, trained by Hugging Face.`: підставляється, коли власного системного тексту немає (як у `simple_qa`).
5. **Порожній блок `<think>\n\n</think>\n`** у режимі `/no_think`: і в репліках асистента з історії (`multi_turn`), і в кінці промпту при `add_generation_prompt=True`
   разом із відкритим заголовком `<|im_start|>assistant\n`.

Практичний висновок для SFT: тренувальний текст має містити ті самі службові частини, що й промпт під час інференсу, інакше модель навчиться на форматі, якого потім не побачить.

### 15. Різниця між Base і Instruct (секція 3)

**Мій вивід на запит `Explain me, what is quantum computing?`** (обидві відповіді обірвались на ліміті `max_new_tokens`):

- **Base** почала не з відповіді, а з **продовження питання**: « And why is it so important?». Лише потім вона перейшла до пояснення в стилі інтернет-статті
  («Quantum computing is the process of using quantum phenomena to perform computation…»), далі почала список застосувань («Drug discovery», «Financial modeling») і обірвалась на півслові.
  Змістовно текст непоганий, але форма — не відповідь співрозмовнику, а продовження документа: модель дописала до нашого запиту «ще одне речення питання» і писала далі так, як роблять веб-тексти.
- **Instruct** одразу відповіла по суті: що таке квантові обчислення, чим кубіти відрізняються від бітів (суперпозиція), де це застосовується. Відповідь має структуру абзаців
  і теж обірвана лише на ліміті токенів («Another key feature of quantum computing is»).

**Висновок.** Base-модель — це передбачення наступного токена за звичайним текстом; вона не знає, що їй поставили *запитання*. Instruct — та сама модель після SFT і вирівнювання
на діалогах: розуміє ролі `user` і `assistant`, відповідає одразу й зв'язно. Також різниться вхід: base отримала сирий текст, instruct — діалог, відформатований шаблоном.
Це головна ідея демо: здатність «вести діалог» не вбудована в модель, її формує донавчання на даних у певному форматі.

### 16. Чому в режимі `/think` передано `max_new_tokens=300`?

У режимі `/think` модель спочатку генерує **міркування** в блоці `<think>…</think>`, і лише потім відповідь. Міркування займає багато токенів, тож стандартні 150 обірвали б генерацію посеред міркувань, і відповіді ми б не побачили. Тому ліміт піднято до 300, проте, як показує вимірювання нижче, навіть цього замало.
`max_new_tokens` — лише верхня межа: якщо модель завершила відповідь раніше (токен кінця репліки), генерація зупиняється сама.
Ліміт варто тримати обмеженим, бо кожен токен коштує часу (і грошей, якщо модель платна).

Клітинка нижче вимірює, скільки токенів реально потрібно в кожному режимі (з запасом до 1000).

In [17]:
import pandas as pd


def generate_len(model, tokenizer, text, max_new_tokens, temperature=0.3):
    """Повертає кількість згенерованих токенів і «сирий» текст (зі службовими токенами)."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return len(new_tokens), tokenizer.decode(new_tokens, skip_special_tokens=False)


rows = []
for flag in ["/no_think", "/think"]:
    for prompt in reasoning_prompts:
        messages = [{"role": "system", "content": flag}, {"role": "user", "content": prompt}]
        formatted = instruct_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        n, raw = generate_len(instruct_model, instruct_tokenizer, formatted, max_new_tokens=1000)
        rows.append({
            "режим": flag,
            "задача": prompt[:45] + "…",
            "токенів згенеровано": n,
            "вмістилось у 300": n <= 300,
            "є </think>": "</think>" in raw,
        })
pd.DataFrame(rows)

,режим,задача,токенів згенеровано,вмістилось у 300,є </think>
0,/no_think,What is 15 × 24? Show your work.…,172,True,False
1,/no_think,A recipe calls for 2 cups of flour for 12 coo…,213,True,False
2,/no_think,If I have $50 and spend $18.75 on lunch and $…,126,True,False
3,/think,What is 15 × 24? Show your work.…,1000,False,True
4,/think,A recipe calls for 2 cups of flour for 12 coo…,1000,False,False
5,/think,If I have $50 and spend $18.75 on lunch and $…,959,False,True


**Мої результати** (ліміт у цьому вимірюванні — 1000 токенів):

| Режим | Задача | Згенеровано токенів | Вмістилось у 300 | Є `</think>` |
|---|---|---|---|---|
| /no_think | 15 × 24 | 172 | так | ні |
| /no_think | борошно для печива | 213 | так | ні |
| /no_think | $50 − обід − кава | 126 | так | ні |
| /think | 15 × 24 | 1000 (ліміт) | **ні** | так |
| /think | борошно для печива | 1000 (ліміт) | **ні** | **ні** |
| /think | $50 − обід − кава | 959 | **ні** | так |

**Висновок.** У режимі `/no_think` відповідь вкладається в 126–213 токенів (у середньому ≈ 170). У режимі `/think` навіть ці прості задачі потребують ≈ 1000 токенів або більше,
тобто щонайменше в 5,8 раза більше, ніж `/no_think`; жодна з трьох не вмістилась би в 300. У задачі про борошно блок міркувань не закрився навіть за 1000 токенів.
Тому в основному прогоні з `max_new_tokens=300` усі три відповіді `/think` обірвались усередині `<think>` і фінальної відповіді не було (див. питання 17).

### 17. Чи впоралася модель із математичними задачами? Який режим обрати?

Правильні відповіді: 15 × 24 = **360**; 5 чашок борошна на 30 печив; 50 − 18,75 − 12,30 = **18,95 $**.

| Режим | 15 × 24 | Борошно для 30 печив | Залишок із $50 |
|---|---|---|---|
| `/no_think` | 360 ✅ | 5 cups ✅ | $18.95 ✅ |
| `/think` (ліміт 300 токенів) | немає відповіді | немає відповіді | немає відповіді |

У режимі `/no_think` модель розв'язала всі три задачі правильно. У режимі `/think` жодна з трьох відповідей не дійшла до кінця: усі обірвались усередині `<think>` (немає `</think>`).
Міркування були на правильному шляху (наприклад, у задачі про гроші модель уже дійшла до 50 − 18,75 = 31,25), але на ліміті 300 токенів відповіді ми не отримали.
Тож це не помилка моделі, а наслідок малого `max_new_tokens`.

**Який режим я б обрав.** Для таких простих двох-трьохкрокових обчислень — `/no_think`: 3 із 3 правильно за 126–213 токенів проти понад 1000 у `/think`, тобто швидше й дешевше.
`/think` я б вмикав для складніших багатокрокових задач або коли помилка коштує дорого, але з набагато більшим `max_new_tokens` (кілька тисяч). Застереження: це три приклади з `temperature=0.3`,
тож робити висновок про надійність режимів лише за ними не можна.

### 18. Які відмінності в шаблонах розмов (секція 5)?

Усі три шаблони передають одні й ті самі ролі `system / user / assistant` (діалог: system «You are a helpful assistant.», user «Hi!», assistant «Hello! How can I help?»), але записують їх по-різному.
Що показав мій вивід:

- **SmolLM3-3B:** ChatML-маркери `<|im_start|>роль … <|im_end|>`. Шаблон сам додав блок `## Metadata` (дата відсічення знань, поточна дата, `Reasoning Mode: /think`: режим за замовчуванням,
  бо прапорець не задавали) і розділ `## Custom Instructions` з нашим системним текстом. Системний блок **не закривається** `<|im_end|>`. У відповіді асистента блоку `<think>` немає.
- **Qwen3-0.6B:** теж ChatML (`<|im_start|>`, `<|im_end|>`), але без метаданих: системний текст лишився як є, `<|im_start|>system\nYou are a helpful assistant.<|im_end|>`.
  Зате шаблон додав до завершеної відповіді асистента **порожній блок** `<think>\n\n</think>\n\n` перед текстом.
- **Phi-3.5-mini-instruct:** інші маркери: `<|system|>`, `<|user|>`, `<|assistant|>`, кінець репліки `<|end|>`, а в самому кінці ще й `<|endoftext|>`.

Отже, відрізняються: набір маркерів ролей і кінця репліки; чи додає шаблон щось «від себе» (метадані в SmolLM3, порожній `<think>` у Qwen3, `<|endoftext|>` у Phi); спосіб керування міркуванням
(прапорець у тексті system у SmolLM3, параметр `enable_thinking` у Qwen3, у Phi немає). Тому шаблон **не можна переносити між моделями**: розмітка вшита в кожну модель окремо.

### 19. Навіщо знати про шаблони розмови, якщо ми хочемо донавчити модель за допомогою SFT?

1. **Формат тренування має збігатися з форматом використання.** Модель вчиться на токенах, а не на «ролях». Якщо навчити її на
   одному шаблоні, а на інференсі подати інший, якість впаде, а відповідь може не закінчуватись (модель не бачила потрібного токена кінця репліки).
2. **Шаблон додає приховані частини** (метадані, режим міркування, порожні блоки `<think>`). Їх треба враховувати в даних: приклад у кінці
   демо показав, що навчання на прикладах без міркувань учить модель відповідати без них навіть у `/think`.
3. **`assistant_only_loss`** потребує знати, де в токенізованому діалозі починається й закінчується відповідь асистента.
4. **Base-модель** часто не має шаблону, або її спеціальні токени ніколи не тренувались; тоді шаблон доведеться вибрати самим
   (найпростіше взяти його з instruct-версії).
5. Від шаблону залежить, **у яку структуру перетворювати власні дані** (`messages`): це визначає, що модель побачить у тренуванні.

### 20. З яких даних побудовано датасет `HuggingFaceTB/smoltalk2`?

SmolTalk2 — відкритий корпус для постнавчання **SmolLM3-3B**. Він має три підмножини, що відповідають трьом етапам навчання
(за карткою датасету на Hugging Face):

- **Mid** (≈4,8 млн рядків) — для проміжного навчання міркуванню. Складається з двох датасетів: Llama-Nemotron-Post-Training-Dataset (3,64 млн рядків)
  і OpenThoughts3-1.2M (1,14 млн рядків).
- **SFT** — суміш приблизно з 25 датасетів, з яких вилучено приклади з тестових наборів, які використовують для оцінювання (decontamination), і приклади з емодзі.
  Серед складових: Tulu 3 SFT Personas IF, hermes-function-calling-v1, Table-GPT, OpenHermes-2.5, OpenThoughts3 (без ланцюжків міркувань),
  LongAlign-64k та інші. Частини мають суфікси `think` / `no_think`: для режимів із міркуванням і без.
  Окремі частини синтетичні: наприклад, `smoltalk_everyday_convs_reasoning_Qwen3_32B_think` згенерувала Qwen3-32B.
- **Preference** (≈447 тис. рядків) — пари «обрана / відхилена відповідь» для вирівнювання (APO): суміш Tulu 3 8B Preference Mixture
  для режиму `no_think` і версія для `think`, де «обрані» відповіді згенерувала Qwen3-32B, а «відхилені» — Qwen3-0.6B.

Отже, це суміш відкритих людських і синтетичних датасетів від різних авторів, а не один первинний корпус.

У самому демо використано лише одну частину SFT-підмножини, `smoltalk_everyday_convs_reasoning_Qwen3_32B_think` (2057 прикладів): це синтетичні повсякденні розмови, де міркування в `<think>` згенерувала Qwen3-32B.

### 21. Які модифікації даних відбуваються в секції «Перетворення власного датасету у формат `messages`»?

Вихідний GSM8K має колонки `question` і `answer`. Функція `to_messages` + `map(batched=True)` робить таке:

1. **Змінює структуру:** дві колонки перетворюються на одну колонку `messages`, у якій список словників `{"role": …, "content": …}`.
2. **Додає роль `system`** з фіксованою інструкцією `You are a math tutor. Solve problems step by step.` — у вихідних даних її не було.
3. **Розкладає пару за ролями:** `question` стає повідомленням `user`, `answer` — повідомленням `assistant`.
4. **Видаляє оригінальні колонки** (`remove_columns`), щоб лишилось тільки `messages`.
5. Працює лише з вибіркою `train[:100]` для швидкості демо.

Сам текст відповідей не змінюється: у GSM8K він містить розрахункові анотації на кшталт `<<48/2=24>>` і рядок `#### 72` у кінці,
тож модель після SFT вчитиме саме такий стиль відповіді. Блоку міркувань у відповідях немає, тому шаблон додає `/think` та метадані, а модель
вчиться відповідати без міркувань: форма даних і шаблон мають узгоджуватися з майбутнім використанням.

У моєму виводі готовий приклад після шаблону SmolLM3 має довжину 145 токенів (метадані, системна інструкція, питання й відповідь разом).